# Data Cleaning - Club Piscine MMM

This notebook extracts and cleans data from the client's Excel files for use in the Marketing Mix Model.

**Files processed:**
1. Rapport de soumissions 2024 & 2025 - Quote requests by product type
2. Budget 2024 & 2025 - Media spend by channel
3. Recap Tableau Medias 2025 - Campaign performance metrics
4. Calendrier Fiscal - Fiscal calendar reference

**Note:** Blank cells represent missing data (client confirmed) - we preserve these as NaN without imputation.

In [ ]:
# Cell 1: Imports and Setup
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Paths - adjust based on where this notebook is run
project_root = Path().cwd().parent if Path().cwd().name == 'notebooks' else Path().cwd()
raw_path = project_root / 'data' / 'raw'
processed_path = project_root / 'data' / 'processed'

# Create processed directory if it doesn't exist
processed_path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")
print(f"Raw data path: {raw_path}")
print(f"Processed data path: {processed_path}")

# List raw files
print("\nRaw files found:")
for f in raw_path.glob('*'):
    print(f"  - {f.name}")

---
## 1. Rapport de Soumissions (Quote Requests) - 2024 & 2025

Extracts monthly quote requests by product type from the RECAP sheet.

**Data location:**
- **2024 file:** Rows 17-28 (January-December)
- **2025 file:** Rows 20-31 (January-December)

**Columns extracted:**
- Piscines Hors Terre (above-ground pools)
- Piscines Creusées (in-ground pools)
- Spas
- Autres Produits (other products)
- Services
- Autre (other)

**Note:** Column positions differ between years - the code handles this automatically.

In [ ]:
# Cell 2: Clean Rapport de Soumissions 2024 & 2025

def clean_soumissions(file_path, year, start_row, end_row):
    """
    Clean the Rapport de Soumissions file.
    
    Extracts monthly quote data from the RECAP sheet.
    
    Columns extracted:
    - Column C (2): Piscines Hors Terre (above-ground pools)
    - Column G (6): Piscines Creusées (in-ground pools)
    - Column K (10): Spas
    - Column O/S (14/18): Autres Produits
    - Column S/W (18/22): Services
    - Column W/AA (22/26): Autre
    
    Parameters:
    - file_path: Path to the Excel file
    - year: The main year of the data (2024 or 2025)
    - start_row: Start row for the monthly data (0-indexed)
    - end_row: End row for the monthly data (0-indexed, exclusive)
    """
    df_raw = pd.read_excel(file_path, sheet_name='RECAP', header=None)
    
    months = ['Janvier', 'Février', 'Mars', 'Avril', 'Mai', 'Juin', 
              'Juillet', 'Août', 'Septembre', 'Octobre', 'Novembre', 'Décembre']
    
    data = []
    for i, row_idx in enumerate(range(start_row, end_row)):
        if i < len(months):
            # Base columns (same for both years)
            row_data = {
                'year': year,
                'month': months[i],
                'month_num': i + 1,
                'piscines_hors_terre': df_raw.iloc[row_idx, 2],   # Column C
                'piscines_creusees': df_raw.iloc[row_idx, 6],     # Column G  
                'spas': df_raw.iloc[row_idx, 10],                 # Column K
            }
            
            # Extended columns - structure differs between years
            if year == 2025:
                # 2025: Col O=Spas de Nage (skip), S=Autres Produits, W=Services, AA=Autre
                row_data['autres_produits'] = df_raw.iloc[row_idx, 18]   # Column S
                row_data['services'] = df_raw.iloc[row_idx, 22]          # Column W
                row_data['autre'] = df_raw.iloc[row_idx, 26]             # Column AA
            else:  # 2024
                # 2024: Col O=Autres Produits, S=Services, W=Autre, AA=Questions (skip)
                row_data['autres_produits'] = df_raw.iloc[row_idx, 14]   # Column O
                row_data['services'] = df_raw.iloc[row_idx, 18]          # Column S
                row_data['autre'] = df_raw.iloc[row_idx, 22]             # Column W
            
            data.append(row_data)
    
    df_clean = pd.DataFrame(data)
    
    # Convert numeric columns - keep NaN as NaN (no imputation per client request)
    numeric_cols = ['piscines_hors_terre', 'piscines_creusees', 'spas', 
                    'autres_produits', 'services', 'autre']
    
    for col in numeric_cols:
        if col in df_clean.columns:
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    
    # Add total column (sum of main product categories)
    main_cols = ['piscines_hors_terre', 'piscines_creusees', 'spas']
    df_clean['total_main_quotes'] = df_clean[main_cols].sum(axis=1, skipna=True)
    
    # Add total including extended categories
    df_clean['total_all_quotes'] = df_clean[numeric_cols].sum(axis=1, skipna=True)
    
    return df_clean

# Process 2024 file - data in rows 17-28 (0-indexed)
soumissions_2024 = clean_soumissions(
    raw_path / 'Rapport de soumissions 2024.xlsx',
    year=2024,
    start_row=17,
    end_row=29
)

# Process 2025 file - data in rows 20-31 (0-indexed)
soumissions_2025 = clean_soumissions(
    raw_path / '+Rapport de soumissions 2025.xlsx',
    year=2025,
    start_row=20,
    end_row=32
)

# Combine both years
soumissions_combined = pd.concat([soumissions_2024, soumissions_2025], ignore_index=True)

print("=" * 70)
print("SOUMISSIONS 2024 (Quote Requests)")
print("=" * 70)
print(soumissions_2024.to_string(index=False))
print(f"\nMissing values: {soumissions_2024.isna().sum().to_dict()}")

print("\n" + "=" * 70)
print("SOUMISSIONS 2025 (Quote Requests)")
print("=" * 70)
print(soumissions_2025.to_string(index=False))
print(f"\nMissing values: {soumissions_2025.isna().sum().to_dict()}")

---
## 2. Budget Media Spend - 2024 & 2025

Extracts monthly media spend by channel type.

**Channels to EXCLUDE (as specified by client):**
- PROGRAMMATIQUE + GÉOCIBLAGE
- AUDIO ET PODCAST  
- ENVOIS POSTAUX

**Channels to GROUP as 'SOCIAL MEDIA':**
- FACEBOOK + INSTAGRAM (PROMO)
- FACEBOOK + INSTAGRAM (PRODUIT)
- PINTEREST
- TIKTOK

**Data location:**
- Row 6: Month headers (NOVEMBRE, DECEMBRE, etc.)
- Column D: Media channel names
- Each month column contains the spend for that month

In [ ]:
# Cell 3: Clean Budget 2024 & 2025

def clean_budget(file_path, year):
    """
    Clean the Budget file and extract monthly media spend.
    
    The budget files have:
    - Row 6: Month headers (NOVEMBRE, DECEMBRE, etc.)
    - Column D (index 3): Media type names
    - Each month's column contains spend data for that media channel
    
    Returns aggregated spend by year, month, and media channel.
    """
    df_raw = pd.read_excel(file_path, sheet_name=0, header=None)
    
    # Channels to exclude (as specified by client)
    exclude_patterns = [
        'PROGRAMMATIQUE',  # Includes GÉOCIBLAGE
        'AUDIO ET PODCAST',
        'ENVOIS POSTAUX'
    ]
    
    # Channels to group as Social Media
    social_media_patterns = [
        'FACEBOOK', 'INSTAGRAM', 'PINTEREST', 'TIKTOK'
    ]
    
    # Skip patterns - headers, totals, subtotals, language breakdowns, subcomponents
    skip_exact = ['FR', 'EN', 'TRADITIONNEL', 'NUMÉRIQUE', 'AUTRES']
    
    skip_contains = [
        'TOTAL', 'DIFFÉRENCE', '% VS', 
        'SEMAINE', 'CAMPAGNE', 'MEDIA', 'COOP', 'PRODUCTION', 'RÉSERVE',
        'CONTINGENCE', 'CIRCULAIRE PAPIER',
        # Subcomponents in 2024 that are already included in parent rows
        'RECHERCHE DE MOTS CLÉS',
        'PREROLL - YOUTUBE',
    ]
    
    # Month mapping - fiscal year starts in November
    months_info = [
        ('NOVEMBRE', 11), ('DECEMBRE', 12), ('JANVIER', 1), ('FEVRIER', 2),
        ('MARS', 3), ('AVRIL', 4), ('MAI', 5), ('JUIN', 6),
        ('JUILLET', 7), ('AOUT', 8), ('SEPTEMBRE', 9), ('OCTOBRE', 10)
    ]
    
    # Find column indices for each month (first occurrence only, in first 70 cols)
    row6 = df_raw.iloc[6, :].tolist()
    month_cols = {}
    for i, val in enumerate(row6[:70]):  # Budget section is first ~70 cols
        if pd.notna(val):
            val_upper = str(val).upper().strip()
            for month_name, month_num in months_info:
                if val_upper == month_name and month_name not in month_cols:
                    month_cols[month_name] = (i, month_num)
                    break
    
    print(f"  Found {len(month_cols)} months for {year}")
    
    # Data rows to process
    data_rows = list(range(10, 37))
    
    media_data = []
    
    for row_idx in data_rows:
        media_name = df_raw.iloc[row_idx, 3]  # Column D
        
        if pd.isna(media_name) or not str(media_name).strip():
            continue
        
        media_name_str = str(media_name).strip()
        media_name_upper = media_name_str.upper()
        
        # Skip exact matches (FR, EN, etc.)
        if media_name_upper in skip_exact:
            continue
        
        # Skip if contains certain patterns (TOTAL, etc.)
        if any(skip in media_name_upper for skip in skip_contains):
            continue
        
        # For 2024, skip the specific "Bannières web" subcomponent
        # (but not "BANNIÈRES WEB - PREMIUM")
        if year == 2024 and media_name_upper == 'BANNIÈRES WEB':
            continue
        
        # Skip excluded channels
        if any(excl.upper() in media_name_upper for excl in exclude_patterns):
            continue
        
        # Categorize channel
        if any(sm.upper() in media_name_upper for sm in social_media_patterns):
            channel_category = 'SOCIAL MEDIA'
        elif 'GOOGLE' in media_name_upper and 'SHOPPING' not in media_name_upper:
            channel_category = 'GOOGLE ADS'  # Normalize Google channels
        else:
            channel_category = media_name_upper
        
        # Extract spend for each month
        for month_name, (col_idx, month_num) in month_cols.items():
            spend = df_raw.iloc[row_idx, col_idx]
            spend_value = pd.to_numeric(spend, errors='coerce')
            
            if pd.notna(spend_value) and spend_value != 0:
                media_data.append({
                    'year': year,
                    'month': month_name,
                    'month_num': month_num,
                    'media_channel': channel_category,
                    'spend': spend_value
                })
    
    df_clean = pd.DataFrame(media_data)
    
    if df_clean.empty:
        return df_clean
    
    # Aggregate by year, month, and channel (to combine social media)
    df_agg = df_clean.groupby(['year', 'month', 'month_num', 'media_channel'], as_index=False)['spend'].sum()
    
    return df_agg

# Process both budget files
print("Processing Budget 2024...")
budget_2024 = clean_budget(raw_path / 'Budget 2024 - REEL au 5 novembre.xlsx', 2024)

print("Processing Budget 2025...")
budget_2025 = clean_budget(raw_path / 'Budget 2025 - 21 août.xlsx', 2025)

# Combine both years
budget_combined = pd.concat([budget_2024, budget_2025], ignore_index=True)

print("\n" + "=" * 60)
print("BUDGET 2024 - Media Spend by Channel")
print("=" * 60)
if not budget_2024.empty:
    # Create pivot table for better visualization
    month_order = ['NOVEMBRE', 'DECEMBRE', 'JANVIER', 'FEVRIER', 'MARS', 'AVRIL', 
                   'MAI', 'JUIN', 'JUILLET', 'AOUT', 'SEPTEMBRE', 'OCTOBRE']
    pivot_2024 = budget_2024.pivot_table(index='media_channel', columns='month', 
                                          values='spend', aggfunc='sum', fill_value=0)
    pivot_2024 = pivot_2024[[m for m in month_order if m in pivot_2024.columns]]
    print(pivot_2024.round(2).to_string())
    print(f"\nTotal spend 2024: ${budget_2024['spend'].sum():,.2f}")
    print(f"Channels: {sorted(budget_2024['media_channel'].unique())}")

print("\n" + "=" * 60)
print("BUDGET 2025 - Media Spend by Channel")
print("=" * 60)
if not budget_2025.empty:
    pivot_2025 = budget_2025.pivot_table(index='media_channel', columns='month', 
                                          values='spend', aggfunc='sum', fill_value=0)
    pivot_2025 = pivot_2025[[m for m in month_order if m in pivot_2025.columns]]
    print(pivot_2025.round(2).to_string())
    print(f"\nTotal spend 2025: ${budget_2025['spend'].sum():,.2f}")
    print(f"Channels: {sorted(budget_2025['media_channel'].unique())}")

---
## 3. Tableau Medias 2025 - Campaign Performance

Extracts campaign-level metrics from the MASTER-TOTAL sheet.

**Columns extracted (as specified by client):**
- B (1): Date début - Campaign start date
- C (2): Date fin - Campaign end date  
- D (3): Média - Media type
- F (5): Station / Support - Media partner/platform
- L (11): Coût total ($ NET) - Total cost
- T (19): Nb occasions (RÉEL) - Actual number of spots
- U (20): Impressions totales (RÉEL) - Actual impressions
- V (21): PEB (RÉEL) - Actual GRPs
- AB (27): Vues complétées - Completed views
- AC (28): Taux de vues - View rate
- AD (29): Clics (RÉEL) - Actual clicks
- AE (30): Taux de clics - Click rate

In [ ]:
# Cell 4: Clean Tableau Medias 2025

def clean_tableau_medias(file_path):
    """
    Clean the Tableau Medias file.
    
    Extracts specific columns by index (safer than by name due to newlines in headers):
    - B (1): Date début
    - C (2): Date fin
    - D (3): Média
    - F (5): Station / Support
    - L (11): Coût total ($ NET)
    - T (19): Nb occasions (RÉEL)
    - U (20): Impressions totales (RÉEL)
    - V (21): PEB (RÉEL)
    - AB (27): Vues complétées
    - AC (28): Taux de vues
    - AD (29): Clics (RÉEL)
    - AE (30): Taux de clics
    """
    df_raw = pd.read_excel(file_path, sheet_name='MASTER-TOTAL', header=0)
    
    # Extract columns by index (safer than by name)
    cols_to_extract = {
        1: 'date_debut',
        2: 'date_fin', 
        3: 'media_type',
        5: 'support',
        11: 'cost_net',
        19: 'occasions_reel',
        20: 'impressions_reel',
        21: 'peb_reel',
        27: 'vues_completees',
        28: 'taux_vues',
        29: 'clics_reel',
        30: 'taux_clics'
    }
    
    # Extract using iloc
    df_clean = df_raw.iloc[:, list(cols_to_extract.keys())].copy()
    df_clean.columns = list(cols_to_extract.values())
    
    # Remove empty rows (where all selected columns are NaN)
    df_clean = df_clean.dropna(how='all')
    
    # Remove rows where both dates are NaN (not actual campaign data)
    df_clean = df_clean.dropna(subset=['date_debut', 'date_fin'], how='all')
    
    # Convert date columns
    df_clean['date_debut'] = pd.to_datetime(df_clean['date_debut'], errors='coerce')
    df_clean['date_fin'] = pd.to_datetime(df_clean['date_fin'], errors='coerce')
    
    # Convert numeric columns - keep NaN as NaN
    numeric_cols = ['cost_net', 'occasions_reel', 'impressions_reel', 'peb_reel',
                    'vues_completees', 'taux_vues', 'clics_reel', 'taux_clics']
    for col in numeric_cols:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    
    # Add derived columns
    df_clean['year'] = df_clean['date_debut'].dt.year
    df_clean['month'] = df_clean['date_debut'].dt.month
    
    # Calculate CPM where possible
    mask = (df_clean['impressions_reel'] > 0) & df_clean['cost_net'].notna()
    df_clean.loc[mask, 'cpm_calculated'] = (df_clean.loc[mask, 'cost_net'] / df_clean.loc[mask, 'impressions_reel']) * 1000
    
    # Reset index
    df_clean = df_clean.reset_index(drop=True)
    
    return df_clean

# Process the file
tableau_medias = clean_tableau_medias(raw_path / 'Recap_Tableau_Medias_2025.xlsx')

print("=" * 60)
print("TABLEAU MEDIAS 2025 - Campaign Performance")
print("=" * 60)
print(f"\nShape: {tableau_medias.shape}")
print(f"Date range: {tableau_medias['date_debut'].min()} to {tableau_medias['date_fin'].max()}")
print(f"Media types: {tableau_medias['media_type'].unique().tolist()}")
print(f"\nTotal cost: ${tableau_medias['cost_net'].sum():,.2f}")
print(f"Total impressions: {tableau_medias['impressions_reel'].sum():,.0f}")
print(f"Total clicks: {tableau_medias['clics_reel'].sum():,.0f}")

print("\n" + "-" * 40)
print("Summary by Media Type:")
print("-" * 40)
summary = tableau_medias.groupby('media_type').agg({
    'cost_net': 'sum',
    'impressions_reel': 'sum',
    'clics_reel': 'sum'
}).round(2)
print(summary.to_string())

print("\n" + "-" * 40)
print("Missing values per column:")
print("-" * 40)
print(tableau_medias.isna().sum())

---
## 4. Calendrier Fiscal (Fiscal Calendar)

Reference table for fiscal calendar mapping. The client's fiscal year starts in November.

**Key columns of interest (as specified by client):**
- G: Trimestre (Quarter) - Note: labeled 'Trimestre Pas bon' but we use 'Trimestre'
- I: Semaine fiscale (Fiscal week number)
- L: Formule (Week description with date range)

**Additional useful columns included:**
- Date, Année, Mois, Nom Mois
- Année fiscale (Fiscal year)
- Semaine débutant le (Week starting date)

In [ ]:
# Cell 5: Clean Calendrier Fiscal

def clean_calendrier_fiscal(file_path):
    """
    Clean the Calendrier Fiscal file.
    
    Extracts key columns for fiscal calendar reference.
    Focus on: Trimestre, Semaine fiscale, Formule
    Plus additional context columns.
    """
    # Read with first row as header
    df_raw = pd.read_excel(file_path, sheet_name='CalendrierFiscal', header=0)
    
    # Select key columns
    columns_to_keep = [
        'Date', 'Année', 'Mois', 'Nom Mois', 'Jour de la semaine',
        'Année fiscale', 'Trimestre', 'Semaine fiscale', 'Formule',
        'Semaine débutant le', 'Ordre du mois fiscal', 'Ordre semaine',
        'MoisFiscal', 'AnnéeNUM', 'Date début semaine'
    ]
    
    # Keep only columns that exist in the dataframe
    columns_to_keep = [c for c in columns_to_keep if c in df_raw.columns]
    df_clean = df_raw[columns_to_keep].copy()
    
    # Ensure Date is datetime
    df_clean['Date'] = pd.to_datetime(df_clean['Date'], errors='coerce')
    
    # Filter to relevant years (covering fiscal years in our data)
    df_clean = df_clean[df_clean['Année'].isin([2021, 2022, 2023, 2024, 2025, 2026])]
    
    return df_clean

# Process the file
calendrier_fiscal = clean_calendrier_fiscal(raw_path / 'CalendrierFiscal.xlsx')

print("=" * 60)
print("CALENDRIER FISCAL (Fiscal Calendar)")
print("=" * 60)
print(f"\nShape: {calendrier_fiscal.shape}")
print(f"Date range: {calendrier_fiscal['Date'].min()} to {calendrier_fiscal['Date'].max()}")
print(f"\nFiscal years: {sorted(calendrier_fiscal['Année fiscale'].unique().tolist())}")
print(f"Quarters: {calendrier_fiscal['Trimestre'].unique().tolist()}")
print(f"Fiscal weeks: {calendrier_fiscal['Semaine fiscale'].min()} to {calendrier_fiscal['Semaine fiscale'].max()}")

print("\n" + "-" * 40)
print("Sample data (first 10 rows):")
print("-" * 40)
print(calendrier_fiscal[['Date', 'Année fiscale', 'Trimestre', 'Semaine fiscale', 'Formule']].head(10).to_string())

print("\n" + "-" * 40)
print("Rows per fiscal year:")
print("-" * 40)
print(calendrier_fiscal['Année fiscale'].value_counts().sort_index())

---
## 5. Save Cleaned DataFrames

In [ ]:
# Cell 6: Save all cleaned dataframes

# Save to CSV (for easy inspection)
soumissions_combined.to_csv(processed_path / 'soumissions_quotes.csv', index=False)
budget_combined.to_csv(processed_path / 'budget_media_spend.csv', index=False)
tableau_medias.to_csv(processed_path / 'tableau_medias_performance.csv', index=False)
calendrier_fiscal.to_csv(processed_path / 'calendrier_fiscal.csv', index=False)

# Save to pickle (for preserving data types)
soumissions_combined.to_pickle(processed_path / 'soumissions_quotes.pkl')
budget_combined.to_pickle(processed_path / 'budget_media_spend.pkl')
tableau_medias.to_pickle(processed_path / 'tableau_medias_performance.pkl')
calendrier_fiscal.to_pickle(processed_path / 'calendrier_fiscal.pkl')

print("=" * 60)
print("FILES SAVED")
print("=" * 60)
print(f"\nSaved to: {processed_path}")
print("\nFiles created:")
for f in processed_path.glob('*'):
    size_kb = f.stat().st_size / 1024
    print(f"  - {f.name} ({size_kb:.1f} KB)")

---
## 6. Data Quality Summary

In [ ]:
# Cell 7: Data Quality Summary

print("=" * 70)
print("DATA QUALITY SUMMARY")
print("=" * 70)

datasets = {
    'Soumissions (Quotes)': soumissions_combined,
    'Budget (Media Spend)': budget_combined,
    'Tableau Medias': tableau_medias,
    'Calendrier Fiscal': calendrier_fiscal
}

for name, df in datasets.items():
    print(f"\n{'-' * 40}")
    print(f"{name}")
    print(f"{'-' * 40}")
    print(f"  Rows: {len(df):,}")
    print(f"  Columns: {len(df.columns)}")
    
    # Missing values
    total_cells = df.size
    missing_cells = df.isna().sum().sum()
    missing_pct = (missing_cells / total_cells) * 100
    print(f"  Missing values: {missing_cells:,} ({missing_pct:.1f}%)")
    
    # Columns with missing values
    missing_cols = df.columns[df.isna().any()].tolist()
    if missing_cols:
        print(f"  Columns with NaN: {missing_cols}")

print("\n" + "=" * 70)
print("NOTE: Missing values (NaN) are intentionally preserved as per client")
print("instructions - they represent data that was not available/collected.")
print("=" * 70)